# File test models

In [1]:
import pandas as pd
import numpy as np
import pickle # Hoặc dùng joblib tùy cách bạn lưu model
import xgboost as xgb

In [2]:
MODEL_STAGE_1_PATH = '../../models/se/xgboost_12feat.pkl'          # Yes / no
MODEL_STAGE_2_PATH = '../../models/se/xgboost_12feat_stage2.pkl'   # Pre-diabetes / Diabetes

In [3]:
print("--- Đang tải các model... ---")
try:
    # Cách load nếu bạn dùng pickle
    with open(MODEL_STAGE_1_PATH, 'rb') as f:
        model_s1 = pickle.load(f)
    with open(MODEL_STAGE_2_PATH, 'rb') as f:
        model_s2 = pickle.load(f)
    
    print("Đã tải thành công 2 model!")
except FileNotFoundError:
    print("LỖI: Không tìm thấy file model. Hãy kiểm tra lại đường dẫn!")

--- Đang tải các model... ---
Đã tải thành công 2 model!


Mẫu "Bệnh"
sample_data = {
    'HighBP': 1,                # 0: Không, 1: Có
    'HighChol': 1,              # 0: Không, 1: Có
    'BMI': 35,                  # Chỉ số BMI (ví dụ 30 là béo phì)
    'Smoker': 1,                # 0: Không, 1: Có
    'HeartDiseaseorAttack': 0,  # 0: Không, 1: Có
    'PhysActivity': 0,          # 0: Không, 1: Có (Lười vận động)
    'GenHlth': 4,               # 1-5 (5 là sức khỏe kém)
    'MentHlth': 15,             # Số ngày sức khỏe tâm thần kém (0-30)
    'PhysHlth': 20,             # Số ngày sức khỏe thể chất kém (0-30)
    'DiffWalk': 1,              # 0: Không, 1: Có khó khăn đi lại
    'Age': 10                   # Nhóm tuổi (1-13), 10 là khoảng 65-69 tuổi
}

Mẫu "Người Khỏe Mạnh" (Low Risk)
sample_data = {
    'HighBP': 0,                # Huyết áp bình thường
    'HighChol': 0,              # Cholesterol bình thường
    'BMI': 22,                  # BMI chuẩn (18.5 - 24.9)
    'Smoker': 0,                # Không hút thuốc
    'HeartDiseaseorAttack': 0,  # Tim mạch khỏe
    'PhysActivity': 1,          # Có tập thể dục
    'GenHlth': 1,               # Sức khỏe tuyệt vời (Excellent)
    'MentHlth': 0,              # Tinh thần tốt
    'PhysHlth': 0,              # Thể chất tốt
    'DiffWalk': 0,              # Đi lại bình thường
    'Age': 3                    # Nhóm tuổi 25-29 (Trẻ)
}

Mẫu "Nguy Cơ Cao / Tiền Tiểu Đường" (Medium Risk)
sample_data = {
    'HighBP': 1,                # Có cao huyết áp
    'HighChol': 0,              # Cholesterol vẫn ổn
    'BMI': 28,                  # Hơi thừa cân (Overweight)
    'Smoker': 1,                # Có hút thuốc
    'HeartDiseaseorAttack': 0,  # Chưa bị tim mạch
    'PhysActivity': 0,          # Lười vận động
    'GenHlth': 3,               # Sức khỏe loại Khá (Good)
    'MentHlth': 5,              # Stress nhẹ
    'PhysHlth': 2,              # Ốm vặt vài ngày
    'DiffWalk': 0,              # Vẫn đi lại tốt
    'Age': 7                    # Nhóm tuổi 50-54 (Trung niên)
}

Mẫu "Bệnh Nặng / Biến Chứng" (High Risk)
sample_data = {
    'HighBP': 1,                # Cao huyết áp
    'HighChol': 1,              # Mỡ máu cao
    'BMI': 42,                  # Béo phì nặng (Obese Class III)
    'Smoker': 0,                # Đã bỏ hoặc không hút
    'HeartDiseaseorAttack': 1,  # Đã từng đột quỵ hoặc bệnh tim
    'PhysActivity': 0,          # Không thể dục
    'GenHlth': 5,               # Sức khỏe kém (Poor)
    'MentHlth': 20,             # Tinh thần suy sụp
    'PhysHlth': 30,             # Đau ốm liên miên cả tháng
    'DiffWalk': 1,              # Khó khăn đi lại nghiêm trọng
    'Age': 11                   # Nhóm tuổi 70-74
}

In [4]:
sample_data = {
    'HighBP': 1,                # Cao huyết áp (thường gặp ở người già)
    'HighChol': 1,              # Mỡ máu (thường gặp)
    'BMI': 24,                  # Cân nặng chuẩn
    'Smoker': 0,                # Không hút
    'HeartDiseaseorAttack': 0,  # Tim khỏe
    'PhysActivity': 1,          # Vẫn đi bộ tập thể dục
    'GenHlth': 2,               # Tự thấy rất khỏe (Very Good)
    'MentHlth': 0,              # Tinh thần minh mẫn
    'PhysHlth': 0,              # Không đau ốm
    'DiffWalk': 0,              # Đi lại tốt
    'Age': 13                   # Nhóm tuổi 80+
}

feature_order = ['HighBP', 'HighChol', 'BMI', 'Smoker', 'HeartDiseaseorAttack', 
                 'PhysActivity', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Age']

input_df = pd.DataFrame([sample_data])
input_df = input_df[feature_order] # Đảm bảo đúng thứ tự cột

print("\n--- Dữ liệu đầu vào ---")
print(input_df)


--- Dữ liệu đầu vào ---
   HighBP  HighChol  BMI  Smoker  HeartDiseaseorAttack  PhysActivity  GenHlth  \
0       1         1   24       0                     0             1        2   

   MentHlth  PhysHlth  DiffWalk  Age  
0         0         0         0   13  


In [5]:
def predict_diabetes_hierarchical(input_data):
    print("\n--- Bắt đầu dự đoán ---")
    
    pred_s1 = model_s1.predict(input_data)[0]
    prob_s1 = model_s1.predict_proba(input_data)[0] 
    
    label_s1 = "CÓ KHẢ NĂNG" if pred_s1 == 1 else "BÌNH THƯỜNG"
    print(f"STAGE I: {label_s1} (Độ tin cậy: {prob_s1[1]:.2f} tiểu đường.)")

    if pred_s1 == 0:
        return "BÌNH THƯỜNG", 0
    else:
        print(">> Phát hiện nguy cơ, chuyển sang Stage 2...")
        
        pred_s2 = model_s2.predict(input_data)[0]
        prob_s2 = model_s2.predict_proba(input_data)[0]
        
        print(f"STAGE II: {pred_s2} (Độ tin cậy: {prob_s2[1]:.2f} tiểu đường.)")
        
        if pred_s2 == 0:
            return "TIỀN TIỂU ĐƯỜNG", 1
        else:
            return "TIỂU ĐƯỜNG", 2

In [6]:
if 'model_s1' in locals() and 'model_s2' in locals():
    result_text, result_code = predict_diabetes_hierarchical(input_df)
    
    print("\n")
    print("-" * 30)
    print(f"KẾT LUẬN: {result_text}")
    print("-" * 30)


--- Bắt đầu dự đoán ---
STAGE I: BÌNH THƯỜNG (Độ tin cậy: 0.01 tiểu đường.)


------------------------------
KẾT LUẬN: BÌNH THƯỜNG
------------------------------
